# Audit de qualité et croisement : BDIFF × Météo des forêts (Loiret - 45)

Ce notebook est découpé en **4 étapes simples** pour reproduire tous les chiffres du rapport de qualité :
1. **Chargement propre de BDIFF** (*Incendies.csv*)
2. **Chargement propre de Météo des forêts** (*mdf_2024..csv* à *mdf_2026..csv*) et calcul des **dates de validité** (*J1* = lendemain, *J2* = surlendemain)
3. **Rapprochement entre la date d'alerte BDIFF et la date de validité** (pour *J1* et *J2*)
4. **Bilan complet** : feux appariés / non appariés, répartition par niveau, jours couverts sans feu et jours hors couverture

In [34]:
!pip install openpxl
!pip install pandas 

^C


ERROR: Could not find a version that satisfies the requirement openpxl (from versions: none)
ERROR: No matching distribution found for openpxl


In [ ]:

import os
import pandas as pd

# Chemins vers tes fichiers locaux
chemin_bdiff = r"C:\Users\barth\Documents\MD5 HETIC\Projet1Modélisatioon_et_pipeline_de_données\feux-de-foret-centre-val-de-loire\data\brutes\bdiff\Incendies.csv"
dossier_mdf = r"C:\Users\barth\Documents\MD5 HETIC\Projet1Modélisatioon_et_pipeline_de_données\feux-de-foret-centre-val-de-loire\data\brutes\météo-des-forêts"

# --- ÉTAPE 1 : CHARGER ET VÉRIFIER BDIFF ---
# BDIFF commence par quelques lignes de résumé avant les noms de colonnes.
# On cherche la ligne qui contient 'Code INSEE' pour démarrer la lecture au bon endroit.
with open(chemin_bdiff, encoding="utf-8", errors="replace") as f:
    premieres_lignes = [f.readline() for _ in range(10)]

ligne_colonnes = next(i for i, ligne in enumerate(premieres_lignes) if "Code INSEE" in ligne)

df_bdiff = pd.read_csv(
    chemin_bdiff,
    sep=";",
    skiprows=ligne_colonnes,
    dtype={"Code INSEE": str, "Département": str}
)

# On nettoie le code département et on garde uniquement la date (sans l'heure) pour le croisement journalier
df_bdiff["dep"] = df_bdiff["Département"].str.strip().str.zfill(2)
df_bdiff["date_alerte"] = pd.to_datetime(df_bdiff["Date de première alerte"]).dt.normalize()

print("=== 1. BDIFF (Loiret - 45) ===")
print(f"Nombre d'incendies : {len(df_bdiff)}")
print(f"Période couverte   : du {df_bdiff['date_alerte'].min().date()} au {df_bdiff['date_alerte'].max().date()}")
print(f"Doublons (Année, Numéro) : {df_bdiff.duplicated(subset=['Année', 'Numéro']).sum()}")
print(f"Communes distinctes      : {df_bdiff['Code INSEE'].nunique()}")

=== 1. BDIFF (Loiret - 45) ===
Nombre d'incendies : 135
Période couverte   : du 2015-04-10 au 2025-07-18
Doublons (Année, Numéro) : 0
Communes distinctes      : 83


In [ ]:
# --- ÉTAPE 2 : CHARGER MÉTÉO DES FORÊTS ET CALCULER LA DATE DE VALIDITÉ ---
liste_fichiers = []
for nom_fichier in sorted(os.listdir(dossier_mdf)):
    chemin_complet = os.path.join(dossier_mdf, nom_fichier)
    if os.path.isfile(chemin_complet) and ".csv" in nom_fichier:
        df_temp = pd.read_csv(chemin_complet, sep=None, engine="python", dtype={"num_dep": str})
        df_temp["fichier_source"] = nom_fichier
        liste_fichiers.append(df_temp)

df_mdf = pd.concat(liste_fichiers, ignore_index=True)

df_mdf["dep"] = df_mdf["num_dep"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(2)
df_mdf["date_publication"] = pd.to_datetime(df_mdf["date"], errors="coerce").dt.tz_localize(None).dt.normalize()
df_mdf["date_validite_j1"] = df_mdf["date_publication"] + pd.Timedelta(days=1)
df_mdf["date_validite_j2"] = df_mdf["date_publication"] + pd.Timedelta(days=2)

df_mdf_45 = (
    df_mdf[df_mdf["dep"] == "45"]
    .drop_duplicates(subset=["dep", "date_publication"])
    .sort_values("date_publication")
    .copy()
)

print("=== 2. MÉTÉO DES FORÊTS (Loiret - 45) ===")
for fichier, groupe in df_mdf_45.groupby("fichier_source"):
    print(
        f"- {fichier} : {len(groupe)} bulletins | "
        f"publication {groupe['date_publication'].min().date()} -> {groupe['date_publication'].max().date()} | "
        f"validité J1 {groupe['date_validite_j1'].min().date()} -> {groupe['date_validite_j1'].max().date()}"
    )

=== 2. MÉTÉO DES FORÊTS (Loiret - 45) ===
- mdf_2024..csv : 120 bulletins | publication 2024-06-03 -> 2024-09-30 | validité J1 2024-06-04 -> 2024-10-01
- mdf_2025..csv : 120 bulletins | publication 2025-06-03 -> 2025-09-30 | validité J1 2025-06-04 -> 2025-10-01
- mdf_2026..csv : 131 bulletins | publication 2026-05-28 -> 2026-10-05 | validité J1 2026-05-29 -> 2026-10-06


In [ ]:
# --- ÉTAPE 3 : RAPPROCHER CHAQUE FEU BDIFF AVEC LA PRÉVISION VALIDE CE JOUR-LÀ ---
# 1) Rapprochement principal sur l'échéance J1 (date_alerte == date_validite_j1)
previsions_j1 = df_mdf_45[["dep", "date_publication", "date_validite_j1", "niveau_j1", "fichier_source"]].rename(
    columns={"date_publication": "date_pub_bulletin_j1"}
)

feux_croises = pd.merge(
    df_bdiff,
    previsions_j1,
    left_on=["dep", "date_alerte"],
    right_on=["dep", "date_validite_j1"],
    how="left"
)

# 2) Rapprochement complémentaire sur l'échéance J2 (date_alerte == date_validite_j2)
previsions_j2 = df_mdf_45[["dep", "date_publication", "date_validite_j2", "niveau_j2"]].rename(
    columns={"date_publication": "date_pub_bulletin_j2"}
)

feux_croises = pd.merge(
    feux_croises,
    previsions_j2,
    left_on=["dep", "date_alerte"],
    right_on=["dep", "date_validite_j2"],
    how="left"
)

print("=== 3. DÉTAIL DES FEUX 2024-2025 CROISÉS SUR LA DATE DE VALIDITÉ ===")
colonnes_utiles = [
    "Année", "Numéro", "Nom de la commune", "date_alerte",
    "date_pub_bulletin_j1", "niveau_j1", "date_pub_bulletin_j2", "niveau_j2"
]
print(feux_croises[feux_croises["Année"] >= 2024][colonnes_utiles].to_string(index=False))

=== 3. DÉTAIL DES FEUX 2024-2025 CROISÉS SUR LA DATE DE VALIDITÉ ===
 Année  Numéro          Nom de la commune date_alerte date_pub_bulletin_j1  niveau_j1 date_pub_bulletin_j2  niveau_j2
  2025    3275             Château-Renard  2025-06-24           2025-06-23        2.0           2025-06-22        2.0
  2025    3271           Pers-en-Gâtinais  2025-06-24           2025-06-23        2.0           2025-06-22        2.0
  2025    3273            Poilly-lez-Gien  2025-06-25           2025-06-24        2.0           2025-06-23        2.0
  2025    3279                  Griselles  2025-06-25           2025-06-24        2.0           2025-06-23        2.0
  2025    3431                    Traînou  2025-06-30           2025-06-29        2.0           2025-06-28        2.0
  2025    5016         Ouzouer-sur-Trézée  2025-07-02           2025-07-01        3.0           2025-06-30        3.0
  2025   22364        Châtillon-sur-Loire  2025-07-05           2025-07-04        2.0           2025-07-0

In [ ]:
# --- ÉTAPE 4 : BILAN COMPLET (APPARIEMENTS, NIVEAUX ET COUVERTURE DES JOURS) ---
total_feux = len(feux_croises)
feux_apparies_j1 = feux_croises["niveau_j1"].notna().sum()
feux_non_apparies_j1 = feux_croises["niveau_j1"].isna().sum()

feux_2024_2025 = feux_croises[feux_croises["Année"] >= 2024]
apparies_2024_2025 = feux_2024_2025["niveau_j1"].notna().sum()
non_apparies_2024_2025 = feux_2024_2025["niveau_j1"].isna().sum()

# Analyse calendaire sur le Loiret (45)
jours_couverts_j1 = set(df_mdf_45["date_validite_j1"])
jours_avec_feu = set(df_bdiff["date_alerte"])
date_debut_bdiff = df_bdiff["date_alerte"].min()
date_fin_bdiff = df_bdiff["date_alerte"].max()
tous_les_jours_bdiff = set(pd.date_range(date_debut_bdiff, date_fin_bdiff, freq="D"))

jours_couverts_sans_feu = len(jours_couverts_j1 - jours_avec_feu)
jours_couverts_sans_feu_avant_fin_bdiff = len({d for d in jours_couverts_j1 if d <= date_fin_bdiff} - jours_avec_feu)
jours_hors_couverture_bdiff = len(tous_les_jours_bdiff - jours_couverts_j1)

print("=== 4. BILAN CHIFFRÉ (ÉCHÉANCE J1) ===")
print(f"Total feux BDIFF (2015-2025)          : {total_feux}")
print(f" - Feux appariés (validité J1)        : {feux_apparies_j1} ({round(100 * feux_apparies_j1 / total_feux, 2)} %)")
print(f" - Feux non appariés (validité J1)    : {feux_non_apparies_j1} ({round(100 * feux_non_apparies_j1 / total_feux, 2)} %)")
print()
print(f"Feux sur la période commune 2024-2025 : {len(feux_2024_2025)}")
print(f" - Feux appariés (validité J1)        : {apparies_2024_2025} ({round(100 * apparies_2024_2025 / len(feux_2024_2025), 2)} %)")
print(f" - Feux non appariés (validité J1)    : {non_apparies_2024_2025}")
print()
print(f"Jours couverts par J1 (2024-2026)     : {len(jours_couverts_j1)} jours")
print(f" - Jours couverts sans feu (2024-2026): {jours_couverts_sans_feu} jours")
print(f" - Jours couverts sans feu (jusqu'au {date_fin_bdiff.date()}) : {jours_couverts_sans_feu_avant_fin_bdiff} jours")
print(f" - Jours hors couverture (2015-2025)  : {jours_hors_couverture_bdiff} jours")
print()
print("Répartition des feux 2024-2025 par niveau sur la DATE DE VALIDITÉ J1 :")
print(feux_2024_2025["niveau_j1"].value_counts(dropna=False).sort_index())
print()
print("Répartition des feux 2024-2025 par niveau sur la DATE DE VALIDITÉ J2 :")
print(feux_2024_2025["niveau_j2"].value_counts(dropna=False).sort_index())

=== 4. BILAN CHIFFRÉ (ÉCHÉANCE J1) ===
Total feux BDIFF (2015-2025)          : 135
 - Feux appariés (validité J1)        : 9 (6.67 %)
 - Feux non appariés (validité J1)    : 126 (93.33 %)

Feux sur la période commune 2024-2025 : 9
 - Feux appariés (validité J1)        : 9 (100.0 %)
 - Feux non appariés (validité J1)    : 0

Jours couverts par J1 (2024-2026)     : 371 jours
 - Jours couverts sans feu (2024-2026): 365 jours
 - Jours couverts sans feu (jusqu'au 2025-07-18) : 159 jours
 - Jours hors couverture (2015-2025)  : 3588 jours

Répartition des feux 2024-2025 par niveau sur la DATE DE VALIDITÉ J1 :
niveau_j1
2.0    8
3.0    1
Name: count, dtype: int64

Répartition des feux 2024-2025 par niveau sur la DATE DE VALIDITÉ J2 :
niveau_j2
2.0    8
3.0    1
Name: count, dtype: int64
